# Gopalan AI: train the breed classifier

Fine-tunes a pretrained **ConvNeXt-Tiny** to recognise the **16 breeds** on the Gopalan AI website
(8 cattle, 8 buffalo) and exports it to **ONNX** for the app.

## Before you press "Run all"

1. **Settings → Accelerator → GPU** (T4 or P100).
2. **Settings → Internet → On** (needs a phone-verified Kaggle account). It's used to fetch the code and the pretrained weights.
3. **Add Input** (right panel) → add both datasets:
   - `lukex9442/indian-bovine-breeds`
   - `atharvadarpude/indian-buffalo-dataset`
4. **Run all.** It takes about 20–30 minutes on a T4.
5. When it finishes, open the **Output** panel (`/kaggle/working/breed_model`) and download
   `breed_classifier.onnx`, `labels.json`, `metrics.json` and the two `.png` charts.

The training code lives in the repo (`ml/gopalan_ml/breed_train.py`) and is unit-tested. This notebook only calls it.

In [ ]:
# ---- Settings ---------------------------------------------------------------
REPO = "https://github.com/AryanSinghh07/Cattle-Scoring-and-Breed-Selection.git"
BRANCH = "main"          # use "feat/4-breed-classifier-notebook" if the PR isn't merged yet
EPOCHS = 15
BATCH_SIZE = 32
MODEL = "convnext_tiny.fb_in22k_ft_in1k"
OUT_DIR = "/kaggle/working/breed_model"
UPLOAD_TO_HF = False     # set True to push the model to Hugging Face (see last cell)
HF_REPO = "thearyansingh/gopalan-breed-classifier"

In [ ]:
# ---- Check GPU, get the code, install missing packages -----------------------
!nvidia-smi -L || echo "No GPU found: turn on Settings -> Accelerator -> GPU"
!rm -rf /kaggle/working/repo && git clone --quiet --depth 1 -b {BRANCH} {REPO} /kaggle/working/repo
# Only what Kaggle doesn't already have (keeps Kaggle's GPU build of PyTorch).
!pip install -q "timm>=1.0" onnx onnxscript onnxruntime

import sys
sys.path.insert(0, "/kaggle/working/repo/ml")
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ---- Find the datasets --------------------------------------------------------
from pathlib import Path
from gopalan_ml.breed_data import scan
from collections import Counter

ROOTS = [Path("/kaggle/input")]
found = scan(ROOTS)
print(f"{len(found)} images found for our 16 breeds")
print(dict(sorted(Counter(i.slug for i in found).items())))
assert len(found) > 3000, "Datasets missing: add both inputs (see the first cell)."

In [ ]:
# ---- Train, test and export ---------------------------------------------------
from gopalan_ml.breed_train import TrainConfig, run

cfg = TrainConfig(model_name=MODEL, epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=4, out_dir=Path(OUT_DIR))
metrics = run(ROOTS, cfg)

In [ ]:
# ---- Results ------------------------------------------------------------------
import pandas as pd
from IPython.display import Image, display

print(f"Test accuracy: {metrics['accuracy']:.1%}   top-3: {metrics['top3_accuracy']:.1%}   macro-F1: {metrics['macro_f1']:.3f}")
answered = metrics["accuracy_when_answered"]
print(f"'Unsure' below {metrics['unsure_threshold']}: answers {metrics['answered_share']:.0%} of photos"
      + (f", {answered:.1%} correct when it answers" if answered is not None else ""))
print("Data:", {k: v for k, v in metrics['data'].items() if k != 'per_class'})
display(pd.DataFrame(metrics["per_breed"]).T.sort_values("f1-score"))
display(Image(f"{OUT_DIR}/confusion_matrix.png"))
display(Image(f"{OUT_DIR}/training_curves.png"))

In [ ]:
# ---- Check the exported ONNX model on a few test photos -----------------------
import csv
import numpy as np
from PIL import Image as PILImage
from gopalan_ml.breed import BreedClassifier

clf = BreedClassifier(f"{OUT_DIR}/breed_classifier.onnx", f"{OUT_DIR}/labels.json")
rows = [r for r in csv.DictReader(open(f"{OUT_DIR}/manifest.csv")) if r["split"] == "test"][::40][:8]
for r in rows:
    pred = clf.predict(np.asarray(PILImage.open(r["path"]).convert("RGB")))
    print(f"true {r['slug']:13s} -> {pred['breed']:13s} ({pred['confidence']:.2f})")

## Optional: upload to Hugging Face

1. Create a **write** token at https://huggingface.co/settings/tokens
2. In Kaggle: **Add-ons → Secrets → Add** a secret named `HF_TOKEN` with that token, and tick it for this notebook.
3. Set `UPLOAD_TO_HF = True` in the settings cell and run the cell below.

The model is uploaded to a **private** repo. Weights are never committed to GitHub.

In [ ]:
if UPLOAD_TO_HF:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=OUT_DIR, repo_id=HF_REPO,
                      allow_patterns=["*.onnx", "labels.json", "metrics.json", "*.png"])
    print("Uploaded to https://huggingface.co/" + HF_REPO)
else:
    print("Skipped (UPLOAD_TO_HF is False)")